In [13]:
import pandas as pd
import numpy as np
import re

In [14]:
appointments = pd.read_csv('appointments.csv')

In [21]:
appointments.head()

,appointment_id,patient_id,provider_id,department_id,scheduled_date,appointment_date,visit_type,wait_days,status,future_no_show_flag
0,A000000001,P0082338,PR001466,D006,2024-11-14,2024-11-14,Urgent,0,Completed,0
1,A000000002,P0003609,PR001137,D031,2024-01-09,2024-01-24,Procedure,15,Completed,0
2,A000000003,P0060334,PR001536,D035,2024-10-09,2024-10-09,New Consult,0,Completed,0
3,A000000004,P0096218,PR001172,D028,2023-10-28,2023-11-07,Urgent,10,Completed,0
4,A000000005,P0093714,PR001723,D044,2024-06-13,2024-07-16,Follow-up,33,No Show,1


In [16]:
# appointment_id

appointments["appointment_id"].astype(str).str.len().value_counts().sort_index()  # checking id length
appointments["appointment_id"].astype(str).str.startswith("A").all()  # checking if id's start with A
appointments["appointment_id"].isnull().sum()  # no null values

# no cleaning needed

np.int64(0)

In [17]:
# patient_id

appointments["patient_id"].isnull().sum() # no null values

# inconsistent format seen in head

def normalize_patient_id(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().upper()

    # Treat common missing values as NA
    if s in ["", "NAN", "NONE", "UNKNOWN"]:
        return pd.NA

    # Extract all digits
    digits = "".join(re.findall(r"\d+", s))

    if digits == "":
        return pd.NA

    # Standardize to P + 7 digits
    return "P" + digits.zfill(7)

appointments['patient_id'] = appointments['patient_id'].apply(normalize_patient_id)


In [18]:
# provider_id

appointments["provider_id"].astype(str).str.len().value_counts().sort_index()  # checking id length
appointments["provider_id"].astype(str).str.startswith("PR").all()  # checking if id's start with A

# length is fine, but not all id's start with PR
appointments["provider_id"].isnull().sum() # 3984 null values

np.int64(3984)

In [19]:
# department_id

appointments["department_id"].astype(str).str.len().value_counts().sort_index()  # checking id lengths
appointments["department_id"].astype(str).str.startswith("D").all()

# length is fine, but not all id's start with D
appointments["department_id"].isnull().sum() # 2968 null values

np.int64(2968)

In [20]:
# scheduled_date

appointments["scheduled_date"] = pd.to_datetime(appointments["scheduled_date"], errors="coerce", format="mixed")

In [22]:
# appointment_date

appointments["appointment_date"] = pd.to_datetime(appointments["appointment_date"], errors="coerce", format="mixed")

In [23]:
# visit_type

appointments["visit_type"].value_counts()  # all values appear correctly formatted
appointments["department_id"].isnull().sum()  # 2968 null values

np.int64(2968)

In [24]:
# wait_days

appointments['wait_days'].min()  # negative values exist
appointments['wait_days'].max()  # max value (73) is logical

appointments.loc[appointments["wait_days"] < 0, "wait_days"] = np.nan  # turn neg values into NA
appointments["department_id"].isnull().sum()  # 2968 null values now exist

np.int64(2968)

In [25]:
# status

appointments["status"].value_counts()  # inconsistent formatting (capitalization, punctutation)

def normalize_status(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().lower()

    # Completed
    if s in ["completed", "complete", "done"]:
        return "Completed"

    # No show
    if "no" in s and "show" in s:
        return "No Show"
    if s == "noshow":
        return "No Show"

    # Cancelled
    if "cancel" in s:
        return "Cancelled"

    # Rescheduled
    if "resched" in s or "re-sched" in s or "moved" in s:
        return "Rescheduled"

    return np.nan

appointments["status"] = appointments["status"].apply(normalize_status)

In [26]:
# future_no_show_flag

appointments["future_no_show_flag"].value_counts()  # only 1's and 0's, all good
appointments["future_no_show_flag"].isnull().sum()  # no null values exist, all good

np.int64(0)

In [29]:
# all cols cleaned
# drop NAs, check shape, & save

appointments_cleaned = appointments.dropna()
appointments_cleaned.shape  # 991061 rows
appointments_cleaned.isna().sum()  # no NAs

appointments_cleaned.to_csv("appointments_cleaned.csv", index=False)